## Custom Middleware

> https://docs.langchain.com/oss/python/langchain/middleware/custom

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### node-style

In [2]:
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str
    age: int = 99

#### 1. 기술적 관점: 에이전트 실행 데이터 비교

| 구분 | **state** (AgentState) | **runtime** (Runtime) |
| :--- | :--- | :--- |
| **성격** | 내부 데이터 (Internal) | 외부 컨텍스트 (External) |
| **주요 데이터** | 대화 기록, 작업 진행 상황, 카운터 등 | 사용자 ID, 환경 변수, 시스템 설정 등 |
| **변경 여부** | 미들웨어 반환값을 통해 업데이트 가능 | 일반적으로 읽기 전용으로 사용 |
| **사용 목적** | "지금까지 무엇을 했는가?"를 파악 | "누가/어떤 환경에서 실행 중인가?"를 파악 |

#### 2. 비유적 관점: 결재 문서 시스템 비교

| 구분 | **state** (문서 데이터) | **runtime** (결재 시스템 환경) |
| :--- | :--- | :--- |
| **저장 위치** | 결재 문서 파일 그 자체에 저장됨 | 회사 전산망 서버 설정에 저장됨 |
| **가변성** | 결재가 진행될수록 도장이 늘어남 | 결재가 진행되어도 회사명은 변하지 않음 |
| **비유적 의미** | "무엇을(What)" 처리하고 있는가? | "어디서/누가(Where/Who)" 처리하는가? |

In [3]:
from langchain.agents.middleware import before_model

@before_model
def log_before_model(state, runtime):
    print("-" * 30)
    print("state:", state)
    print("runtime", runtime)
    print("-" * 30)
    return None

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[],
    middleware=[log_before_model],
    context_schema=Context
)

In [5]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남")
)

------------------------------
state: {'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='15a2b6c8-570a-43ad-81d9-be90f6ed2daf')]}
runtime Runtime(context=Context(user_name='김일남', age=99), store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x000002AB70260680>, previous=None)
------------------------------


{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='15a2b6c8-570a-43ad-81d9-be90f6ed2daf'),
  AIMessage(content='저는 인공지능이기 때문에 당신의 이름을 알 수 없습니다. 저는 개인 정보를 저장하거나 접근할 수 없거든요.\n\n하지만 만약 원하신다면 저에게 이름을 알려주셔도 괜찮습니다. 그러면 이 대화 중에 당신을 그 이름으로 불러드릴 수 있어요! 😊', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d858e-d43f-7af0-91eb-7b32747a2437-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 609, 'total_tokens': 615, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 551}})]}

### wrap-style

> context에 스키마를 전달하는 것 ≠ LLM이 그 정보를 아는 것

context는 미들웨어/노드 간 데이터를 공유하는 런타임 저장소입니다.<br>
LLM이 해당 정보를 활용하게 하려면,<br>미들웨어에서 request.override(system_prompt=...)처럼 명시적으로 프롬프트에 주입해야 합니다.

In [ ]:
# from langchain.agents.middleware import wrap_model_call
# from langchain.messages import HumanMessage, SystemMessage

# @wrap_model_call
# def inject_user_name(request, handler):
#     print(f"request: {request}")
#     print("-" * 10)
#     return handler(request)


In [6]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, SystemMessage

@wrap_model_call
def inject_user_name(request, handler):
    print(f"request: {request}")
    print("-" * 10)

    user_name =request.runtime.context.user_name
    
    if user_name:
        sys_prompt = f"사용자의 이름은 {user_name}입니다."
    else:
        sys_prompt = "사용자의 이름은 알려지지 않았습니다."
    request = request.override(system_prompt=sys_prompt)
    
    return handler(request)

In [7]:
from langchain.agents.middleware import after_model

@after_model
def log_after_model(state, runtime):
    print(f"after_model_state: {state}")
    print("-" * 10)
    return None

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[],
    middleware=[inject_user_name, log_after_model],
    context_schema=Context
)

In [9]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남")
)

request: ModelRequest(model=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000002AB70135190>, default_metadata=(), model_kwargs={}), messages=[HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='072304ea-7894-435a-bf77-8c86f64c5664')], system_message=None, tool_choice=None, tools=[], response_format=None, state={'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='072304ea-7894-435a-bf77-8c86f64c5664')]}, runtime=Runtime(context=Context(user_name='김일남', age=99

{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='072304ea-7894-435a-bf77-8c86f64c5664'),
  AIMessage(content='김일남입니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d8591-7158-7f60-9e58-7b48ad654b1f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 54, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 49}})]}